In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/marziyeazad/corpus-test/test.jsonl
/kaggle/input/datasets/marziyeazad/corpus-test/corpus.jsonl


In [2]:
!pip install torch transformers accelerate faiss-cpu rank-bm25 bert-score tqdm numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
"""
 Hybrid Retrieval + Cross-Encoder Reranking + RAG (T5 VERSION)

✔ Dense retrieval: FAISS
✔ Lexical retrieval: BM25
✔ Hybrid: 20 FAISS + 20 BM25 → Cross-Encoder rerank → top 10
✔ Answering model: FLAN-T5-small
✔ Judge-as-LLM: FLAN-T5-Large
✔ Evaluation: BERTScore
✔ Option B: Strong prompt to avoid empty answers
"""

from __future__ import annotations
import os, re, json, hashlib
from dataclasses import dataclass
from typing import List

import torch
from tqdm import tqdm
import faiss
from rank_bm25 import BM25Okapi
from bert_score import score as bertscore
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
import numpy as np

# ============================================================
# CONFIG
# ============================================================

CORPUS_JSONL = "/kaggle/input/datasets/marziyeazad/corpus-test/corpus.jsonl"
TEST_JSONL   = "/kaggle/input/datasets/marziyeazad/corpus-test/test.jsonl"

OUT_DIR = "outputs_hw3"
os.makedirs(OUT_DIR, exist_ok=True)

CHUNK_TOKENS = 250
OVERLAP = 40

TOPK_SINGLE = 10   # single FAISS/BM25
TOPK_HYBRID = 20   # candidate pool per retriever for hybrid
TOPK_FINAL  = 10   # final reranked size

BERT_LANG = "en"

EMBED_MODEL = "BAAI/bge-large-en-v1.5"
ANSWER_MODEL = "google/flan-t5-small"
JUDGE_MODEL  = "google/flan-t5-large"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

INSUFFICIENT = "The provided sources do not contain enough information to answer the question"

# ============================================================
# DATA HELPERS
# ============================================================

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def stable_hash(t):
    return hashlib.sha1(t.encode()).hexdigest()[:10]

# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_table_text(text: str) -> str:
    """
    Preprocess table-heavy text for retrieval and embedding.

    Steps:
    1. Keep table structure using | for columns and \n for rows.
    2. Remove HTML tags like <td>, <tr>, <th> but preserve structure.
    3. Normalize ISBNs and punctuation.
    4. Tokenize Unicode words (keeps [UNK] or other symbols safe).
    5. Collapse multiple spaces.
    6. Lowercase everything for BM25 consistency.
    """

    # 1️⃣ Replace row and column tags with delimiters
    text = re.sub(r"</tr\s*>", "\n", text, flags=re.IGNORECASE)  # end of row -> newline
    text = re.sub(r"</td\s*>", " | ", text, flags=re.IGNORECASE) # end of cell -> pipe
    text = re.sub(r"<(td|tr|th)[^>]*>", "", text, flags=re.IGNORECASE)  # remove opening tags

    # 2️⃣ Normalize ISBNs (remove spaces, dashes, unify case)
    text = re.sub(
        r"isbn\s*[-–—]?\s*(\d[\d\s-]+)",
        lambda m: f"ISBN{re.sub(r'[\s-]', '', m.group(1))}",
        text,
        flags=re.IGNORECASE
    )

    # 3️⃣ Normalize other punctuation
    text = text.replace("⁄", "/")  # fractions
    text = text.replace("&", "and")  # optional
    text = re.sub(r"[^\w\s|]", " ", text, flags=re.UNICODE)  # remove unwanted punctuation, keep pipes

    # 4️⃣ Collapse multiple spaces and normalize pipes
    text = re.sub(r"\s*\|\s*", " | ", text)  # normalize spacing around pipes
    text = re.sub(r"\s+", " ", text)  # collapse multiple spaces
    text = text.strip()

    # 5️⃣ Lowercase for BM25 consistency
    text = text.lower()

    return text

# ============================================================
# CHUNKING
# ============================================================

def sent_split(t):
    return re.split(r"(?<=[.!?])\s+", t.strip())

@dataclass
class Chunk:
    id: str
    text: str

def chunk_corpus(corpus, tokenizer):
    chunks = []
    step = CHUNK_TOKENS - OVERLAP

    for doc in tqdm(corpus, desc="Chunking"):
        # Preprocess before chunking
        clean_text = preprocess_table_text(doc["text"])
        sents = sent_split(clean_text)
        packed, cur, cur_len = [], [], 0

        for s in sents:
            l = len(tokenizer(s, add_special_tokens=False)["input_ids"])
            if cur_len + l <= CHUNK_TOKENS:
                cur.append(s)
                cur_len += l
            else:
                packed.append(" ".join(cur))
                cur, cur_len = [s], l
        if cur:
            packed.append(" ".join(cur))

        ids = tokenizer(" ".join(packed), add_special_tokens=False)["input_ids"]
        for i, st in enumerate(range(0, len(ids), step)):
            sub = ids[st:st+CHUNK_TOKENS]
            txt = tokenizer.decode(sub).strip()
            if txt:
                chunks.append(Chunk(
                    id=f"{doc['id']}__{i}__{stable_hash(txt)}",
                    text=txt
                ))
    print('len(chunks)', len(chunks))
    # ---------- saving chunks in a JSONL file ----------
    out_file = "corpus-chunks.jsonl"
    with open(out_file, "w", encoding="utf-8") as f:
        for c in chunks:
            json.dump({"id": c.id, "text": c.text}, f, ensure_ascii=False)
            f.write("\n")
    print(f"Chunks saved to {out_file}")
    return chunks

# ============================================================
# EMBEDDING + FAISS
# ============================================================

class Embedder:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
        self.model = AutoModel.from_pretrained(
            EMBED_MODEL, torch_dtype=DTYPE
        ).to(DEVICE).eval()

    @torch.no_grad()
    def encode(self, texts, bs=128):
        vecs = []
        for i in tqdm(range(0, len(texts), bs), desc="Encoding"):
            enc = self.tok(
                texts[i:i+bs],
                padding=True,
                truncation=False,
                return_tensors="pt"
            ).to(DEVICE)
            out = self.model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1)
            pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
            pooled = torch.nn.functional.normalize(pooled.to(torch.float32), dim=1)
            vecs.append(pooled.cpu())
        return torch.cat(vecs)

    def encode_query(self, q):
        return self.encode([q]).numpy()

def build_faiss(vectors):
    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors.numpy())
    return idx

def retrieve_faiss(q, emb, idx, chunks, k):
    _, I = idx.search(emb.encode_query(q), k)
    return [chunks[i] for i in I[0]]

# ============================================================
# BM25 RETRIEVAL
# ============================================================

def build_bm25(chunks):
    # Tokenize Unicode words for BM25
    tok = lambda t: re.findall(r"\w+", t.lower(), flags=re.UNICODE)
    return BM25Okapi([tok(c.text) for c in chunks])

def retrieve_bm25(q, bm25, chunks, k):
    tok = re.findall(r"\w+", q.lower(), flags=re.UNICODE)
    scores = bm25.get_scores(tok)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in idx]

# ============================================================
# CROSS-ENCODER RERANKER
# ============================================================

class CrossEncoderReranker:
    def __init__(self, name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            name
        ).to(DEVICE).eval()

    @torch.no_grad()
    def score(self, query, passages, batch_size=32):
        scores = []
        for i in range(0, len(passages), batch_size):
            batch = passages[i:i+batch_size]
            inputs = self.tok(
                [query]*len(batch),
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(DEVICE)
            outputs = self.model(**inputs)
            batch_scores = outputs.logits.squeeze(-1)
            scores.extend(batch_scores.cpu().tolist())
        return scores

# ============================================================
# T5 MODEL
# ============================================================

class T5:
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            name, torch_dtype=DTYPE
        ).to(DEVICE).eval()

    @torch.no_grad()
    def gen(self, prompt, max_new=128):
        inp = self.tok(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(DEVICE)
        out = self.model.generate(
            **inp,
            max_new_tokens=max_new,
            num_beams=4,
            do_sample=False
        )
        return self.tok.decode(out[0], skip_special_tokens=True)

# ============================================================
# PROMPTS (Option B: Strong, explicit fallback)
# ============================================================

def rag_prompt(q, chs):
    ctx = "\n".join(c.text for c in chs)
    return f"""
Answer the question using only the provided context.
If the answer is NOT present in the context, you MUST respond exactly with:
"{INSUFFICIENT}"

Context:
{ctx}

Question:
{q}

Answer:
"""

def judge_prompt(q, gold, pred):
    return f"""
Score how similar the model answer is to the gold answer.
Output a single number between 0 and 1.

Question: {q}
Gold: {gold}
Prediction: {pred}

Score:
"""

# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    corpus = read_jsonl(CORPUS_JSONL)
    test   = read_jsonl(TEST_JSONL)

    # Chunk corpus (preprocessing happens inside chunk_corpus)
    tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
    chunks = chunk_corpus(corpus, tok)

    # Encode chunks
    emb = Embedder()
    vecs = emb.encode([c.text for c in chunks])
    vecs_np = vecs.numpy()
    chunkid_to_idx = {c.id: i for i, c in enumerate(chunks)}

    # Build retrievers
    faiss_idx = build_faiss(vecs)
    bm25 = build_bm25(chunks)

    # Load models
    answerer = T5(ANSWER_MODEL)
    judge = T5(JUDGE_MODEL)
    reranker = CrossEncoderReranker()

    preds = {"faiss": [], "bm25": [], "hybrid": []}
    golds = []
    judge_scores = {"faiss": [], "bm25": [], "hybrid": []}

    for ex in tqdm(test, desc="Answering"):
        q = ex["question"]
        gold = ex["answers"][0] if isinstance(ex["answers"], list) else ex["answers"]
        golds.append(gold)

        # --------------------------
        # SINGLE RETRIEVAL (10)
        # --------------------------
        fa = retrieve_faiss(q, emb, faiss_idx, chunks, TOPK_SINGLE)
        bm = retrieve_bm25(q, bm25, chunks, TOPK_SINGLE)

        # --------------------------
        # HYBRID RETRIEVAL (20 + 20)
        # --------------------------
        fa_h = retrieve_faiss(q, emb, faiss_idx, chunks, TOPK_HYBRID)
        bm_h = retrieve_bm25(q, bm25, chunks, TOPK_HYBRID)

        hy_candidates = list({c.id: c for c in fa_h + bm_h}.values())

        # Cross-encoder scoring
        texts = [c.text for c in hy_candidates]
        scores = reranker.score(q, texts)
        sorted_idx = np.argsort(-np.array(scores))
        hy = [hy_candidates[i] for i in sorted_idx[:TOPK_FINAL]]

        # --------------------------
        # Generate answers
        # --------------------------
        for name, ctx in [("faiss", fa), ("bm25", bm), ("hybrid", hy)]:
            ans = answerer.gen(rag_prompt(q, ctx)).strip()
            if not ans:
                ans = INSUFFICIENT
            preds[name].append(ans)

            j = judge.gen(judge_prompt(q, gold, ans), max_new=16)
            m = re.search(r"(0(?:\.\d+)?|1(?:\.0+)?)", j)
            judge_scores[name].append(float(m.group(1)) if m else 0.0)

    # --------------------------
    # Evaluation
    # --------------------------
    for k in preds:
        clean_preds = [p if p.strip() else INSUFFICIENT for p in preds[k]]
        clean_golds = [g if g.strip() else "N/A" for g in golds]

        P, R, F = bertscore(clean_preds, clean_golds, lang=BERT_LANG, verbose=False)
        print(f"\n{k.upper()} BERTScore:",
              {"P": float(P.mean()),
               "R": float(R.mean()),
               "F1": float(F.mean())})
        print(f"{k.upper()} Judge score:",
              sum(judge_scores[k])/len(judge_scores[k]))

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BM25 BERTScore: {'P': 0.7873916029930115, 'R': 0.8439235091209412, 'F1': 0.814018726348877}
BM25 Judge score: 0.7763333333333333


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



HYBRID BERTScore: {'P': 0.7894100546836853, 'R': 0.8491398096084595, 'F1': 0.8174614310264587}
HYBRID Judge score: 0.8116666666666666
